<a href="https://colab.research.google.com/github/vijayrajeshr/Flipkart__GridLockHackathon2.0/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flipkart Gridlock Hackathon 2.0

In [13]:
!pip install pygeohash lightgbm catboost xgboost pandas numpy

In [21]:
import pandas as pd
import numpy as np
import pygeohash as pgh
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

print("1. Loading datasets...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
code2 = pd.read_csv('CODE2.csv') # Your 88% anchor

# --- THE GRANDMASTER SECRET: LOG-TRANSFORM THE TARGET ---
print("2. Applying Logarithmic Compression to Target...")
train['log_demand'] = np.log1p(train['demand'])

print("3. Engineering Multiplicative Physics...")
global_mean = train['log_demand'].mean()

# Base Location Demand (Using Log Space)
gh_base = train.groupby('geohash')['log_demand'].mean().to_dict()
# Time-of-Day Multiplier
time_mean = train.groupby('timestamp')['log_demand'].mean()
time_mult = (time_mean / global_mean).to_dict()
# Weather Multiplier
weather_mean = train.groupby('Weather')['log_demand'].mean()
weather_mult = (weather_mean / global_mean).to_dict()

def get_physics_baseline(row):
    base = gh_base.get(row['geohash'], global_mean)
    t_m = time_mult.get(row['timestamp'], 1.0)
    w_m = weather_mult.get(row['Weather'], 1.0)
    return base * t_m * w_m

train['physics_baseline'] = train.apply(get_physics_baseline, axis=1)
test['physics_baseline'] = test.apply(get_physics_baseline, axis=1)

print("4. Preparing Spatial-Temporal Features...")
def prep_features(df):
    df = df.copy()
    df['lat'] = df['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notna(x) else 0)
    df['lon'] = df['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notna(x) else 0)
    df['hour'] = df['timestamp'].astype(str).apply(lambda x: int(x.split(':')[0]) if pd.notna(x) and ':' in x else 0)
    df['min'] = df['timestamp'].astype(str).apply(lambda x: int(x.split(':')[1]) if pd.notna(x) and ':' in x else 0)
    df['time_of_day'] = df['hour'] * 60 + df['min']
    return df

train_df = prep_features(train)
test_df = prep_features(test)

cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
for col in cat_cols:
    train_df[col] = train_df[col].fillna('Unknown').astype(str)
    test_df[col] = test_df[col].fillna('Unknown').astype(str)

train_df['Temperature'] = train_df['Temperature'].fillna(train_df['Temperature'].median())
test_df['Temperature'] = test_df['Temperature'].fillna(train_df['Temperature'].median())
train_df['NumberofLanes'] = train_df['NumberofLanes'].fillna(train_df['NumberofLanes'].mode()[0])
test_df['NumberofLanes'] = test_df['NumberofLanes'].fillna(train_df['NumberofLanes'].mode()[0])

features = ['physics_baseline', 'lat', 'lon', 'time_of_day', 'Temperature', 'NumberofLanes'] + cat_cols

X = train_df[features]
# WE TRAIN THE MODEL ON THE LOGARITHMIC DEMAND
y = train_df['log_demand']
X_test = test_df[features]

print("5. Training Dual AI Engines on Log-Space...")
# Engine 1: CatBoost
cb = CatBoostRegressor(iterations=1500, learning_rate=0.035, depth=8, cat_features=cat_cols, random_seed=42, verbose=0)
cb.fit(X, y)
cb_log_preds = cb.predict(X_test)

# Engine 2: XGBoost (Categoricals handled natively)
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')
xgb = XGBRegressor(n_estimators=1200, learning_rate=0.035, max_depth=8, tree_method='hist', enable_categorical=True, random_state=42, n_jobs=-1)
xgb.fit(X, y)
xgb_log_preds = xgb.predict(X_test)

print("6. Decompressing Predictions and Applying Hard Ceilings...")
# We decompress the predictions back to real-world traffic numbers
cb_preds = np.expm1(cb_log_preds)
xgb_preds = np.expm1(xgb_log_preds)

# 40% CatBoost + 40% XGBoost + 20% CODE2 Anchor (Your 88% Safety Net)
final_demand = (cb_preds * 0.40) + (xgb_preds * 0.40) + (code2['demand'].values * 0.20)

# --- THE GRANDMASTER R2 PROTECTOR: HARD CEILINGS ---
# Find the absolute max demand for every road in history
gh_max_demand = train.groupby('geohash')['demand'].max().to_dict()
global_max = train['demand'].max()

def apply_ceiling(row, pred):
    road_max = gh_max_demand.get(row['geohash'], global_max)
    # We allow a 15% variance for extreme weather, but absolutely forbid wild over-predictions
    hard_cap = road_max * 1.15
    return min(pred, hard_cap)

# Apply the mathematical ceiling to protect the R2 score
test_df['final_demand'] = final_demand
test_df['clipped_demand'] = test_df.apply(lambda row: apply_ceiling(row, row['final_demand']), axis=1)

# Final safety clip for negative numbers
final_clipped_demand = np.clip(test_df['clipped_demand'], 0, None)

sub = pd.DataFrame({
    'Index': test_df['Index'],
    'demand': final_clipped_demand
})

filename = 'ULTIMATE_97_LOG_CLIPPED.csv'
sub.to_csv(filename, index=False)
print(f"SUCCESS! Download '{filename}'.")

1. Loading datasets...
2. Applying Logarithmic Compression to Target...
3. Engineering Multiplicative Physics...
4. Preparing Spatial-Temporal Features...
5. Training Dual AI Engines on Log-Space...
6. Decompressing Predictions and Applying Hard Ceilings...
SUCCESS! Download 'ULTIMATE_97_LOG_CLIPPED.csv'.
